# Introduction to Generative Models

A **generative model** is a probabilistic model for data. This sentence is short, but it contains the whole philosophy of the field. If images are viewed as random variables taking values in a high-dimensional space, then a generative model aims to describe the law according to which these images occur. Once such a law has been learned, new images can be sampled, partial information can be completed, missing content can be inferred, and prior knowledge can be injected into downstream tasks. In other words, generation is not only about synthesizing visually pleasing pictures. It is about learning a distribution rich enough to support reasoning.

Let $\boldsymbol{x} \in \mathbb{R}^d$ denote an image and let $p_{gt}(\boldsymbol{x})$ denote the unknown data-generating distribution. A learned model introduces a family $\{p_\theta(\boldsymbol{x}) : \theta \in \Theta\}$ and tries to choose parameters $\theta$ so that $p_\theta$ is close to $p_{gt}$. The central technical problem is that $p_{gt}$ is never known analytically. We only observe samples $\boldsymbol{x}^{(1)}, \ldots, \boldsymbol{x}^{(n)}$. Deep generative modeling is therefore the study of how to represent and fit complicated distributions using neural networks from finite data.

This chapter sits between the neural-network preliminaries and the probabilistic foundations for a reason. Once we know what kinds of function classes modern deep networks provide, the next question is what those functions are supposed to represent in a generative setting. Before we can talk about latent variables, variational bounds, or diffusion paths, we need a clear picture of the target problem itself. What does it mean to learn a distribution rather than a classifier. Why is image generation harder than label prediction. Which criteria distinguish model families at a high level. Those are the questions this chapter is meant to stabilize.

One reason this problem is harder than it may first appear is that high-dimensional image space is dominated by configurations that do not look like natural images at all. If one samples pixel values independently at random, the result is almost always meaningless noise. Natural images form a tiny, highly structured subset of the ambient space, with strong dependencies across locations, scales, and semantic content. A generative model is successful only when it captures these dependencies well enough that fresh samples fall near the same structured region as the data. The challenge is therefore geometric as much as statistical.

This is also the right place to state an important philosophical point for the course. A generative model is not merely a machine for producing visually plausible outputs. It is a **hypothesis about how data are organized**. When we choose latent variables, adversarial critics, denoising paths, or transport fields, we are choosing not only an optimization objective but also a language for describing image distributions. Different model families correspond to different languages.

## Generative Versus Discriminative Learning

Discriminative models aim at conditional tasks such as predicting a label $y$ from an image $\boldsymbol{x}$. In probabilistic language, they approximate objects like $p(y | \boldsymbol{x})$ or decision rules derived from it. Generative models target $p(\boldsymbol{x})$ itself, or a joint distribution $p(\boldsymbol{x}, \boldsymbol{z})$ involving hidden variables. This difference changes the geometry of the task. The image space is enormous, multimodal, and structured. Modeling its full law is more ambitious than solving a supervised prediction problem, but it also provides more flexibility.

A deep generative model is therefore not merely a classical generative model with a large parameter count. The adjective deep signals that the parameterization is compositional and learned by neural networks. The decoder of a VAE, the generator of a GAN, the score network of a diffusion model, and the velocity field of a flow matching model are all examples of neural parameterizations that transform a simple distribution into a complicated one.

It is often helpful to contrast the information demanded by the two paradigms. A discriminative classifier may succeed even if it learns only the features necessary to separate categories. It does not need to know how to generate the full image, nor how likely one background texture is relative to another when both correspond to the same label. A generative model, by contrast, must account for the whole observation. It must model nuisance variation, texture, shape, layout, and all the dependencies that make an image look coherent. This is why generative learning is typically harder, but also why it can be more powerful once successful.

For students who are less familiar with the machine-learning distinction, one simple metaphor can help. A discriminative model acts like an examiner who sees an answer sheet and decides which class it belongs to. A generative model acts like an author who must know how to produce a plausible answer sheet in the first place. The second task contains much richer structural information.

## Why Likelihood Matters, and Why It Is Not the Whole Story

If a model has a tractable density $p_\theta(\boldsymbol{x})$, a natural learning principle is maximum likelihood: $\theta^\star \in \arg\max_\theta \sum_{i=1}^n \log p_\theta(\boldsymbol{x}^{(i)})$. This objective has a clear statistical interpretation. Under standard assumptions, maximizing empirical log-likelihood amounts to minimizing the Kullback-Leibler divergence from the data distribution to the model family up to an additive constant. The next theorem makes this precise.

```{prf:theorem} Maximum likelihood and forward Kullback-Leibler divergence
:label: thm-mle-kl

Let $p_{gt}$ be the true data distribution and let $p_\theta$ be any model distribution such that $\log p_\theta(\boldsymbol{x})$ is integrable under $p_{gt}$. Then
:::{math}
\operatorname{KL}(p_{gt} \| p_\theta)
=
\mathbb{E}_{p_{gt}}[\log p_{gt}(\boldsymbol{x})]
-
\mathbb{E}_{p_{gt}}[\log p_\theta(\boldsymbol{x})].
:::
Consequently, minimizing $\operatorname{KL}(p_{gt} \| p_\theta)$ over $\theta$ is equivalent to maximizing the expected log-likelihood under the data distribution.
```

```{prf:proof}
By definition,
:::{math}
\operatorname{KL}(p_{gt} \| p_\theta)
=
\mathbb{E}_{p_{gt}}
\left[
\log \frac{p_{gt}(\boldsymbol{x})}{p_\theta(\boldsymbol{x})}
\right].
:::
Splitting the logarithm gives
:::{math}
\operatorname{KL}(p_{gt} \| p_\theta)
=
\mathbb{E}_{p_{gt}}[\log p_{gt}(\boldsymbol{x})]
-
\mathbb{E}_{p_{gt}}[\log p_\theta(\boldsymbol{x})].
:::
The first term does not depend on $\theta$, so the minimizer of the divergence coincides with the maximizer of the second term. Replacing the population expectation with the empirical average yields the usual maximum likelihood estimator.
```

The previous argument is elegant, but in deep generative modeling exact likelihood is often unavailable or inconvenient. This is why different model families choose different compromises. Variational autoencoders keep a likelihood but optimize a lower bound because exact inference over latent variables is hard. GANs abandon explicit likelihood and train through adversarial discrimination. Diffusion models recover tractable objectives by decomposing generation into many small denoising steps. Flow matching learns vector fields that transport a simple distribution to the data distribution along a chosen probability path.

This variety of strategies is one of the reasons the field can feel fragmented at first. Students often ask whether VAEs, GANs, and diffusion models are competitors, unrelated inventions, or successive improvements of one universal idea. The right answer is more subtle. They are all trying to approximate the unknown data distribution, but they do so by making different tradeoffs between tractable probability, optimization stability, sample quality, and computational cost. A VAE gives us a transparent probabilistic story and amortized inference, but may blur samples. A GAN may produce sharp outputs, but its optimization is delicate and its density is implicit. A diffusion model is often stable and high quality, but sampling may be slow. Flow matching tries to retain the continuous-time transport viewpoint while simplifying training.

Once these tradeoffs are visible, the course becomes easier to navigate. One is not memorizing isolated acronyms, but studying different answers to the same foundational problem.

## A First Taxonomy

One can organize the field along two axes. The first axis asks whether the model uses latent variables. The second asks whether the density is evaluated directly, indirectly, or not at all. A VAE introduces a latent variable $\boldsymbol{z}$ and writes $p_\theta(\boldsymbol{x}) = \int p_\theta(\boldsymbol{x} | \boldsymbol{z}) p(\boldsymbol{z}) \, d\boldsymbol{z}$, but the integral is usually intractable, so learning proceeds by variational inference. A GAN maps a simple latent noise vector to an image through a generator, but the induced density is implicit, meaning that sampling is easy while density evaluation is not. A diffusion model introduces a latent path rather than a single latent code, and its training objective emerges from the controlled corruption and reconstruction of data. Flow matching focuses on continuous transport and learns a deterministic vector field that induces the desired terminal distribution.

```{figure} ../assets/images/DLVM.png
:width: 68%
:align: center

The latent-variable viewpoint is one of the recurring conceptual templates of the course.
```

It is also worth emphasizing that the taxonomy is not rigid. Many modern systems are hybrid. Latent diffusion models combine autoencoding ideas with diffusion in latent space. Conditional GANs and diffusion models add side information and no longer model only a plain marginal distribution. Normalizing flows provide exact likelihoods but are themselves continuous transport models. The purpose of the taxonomy is therefore not to create boxes that every method must fit perfectly. Its purpose is to help the student identify the main design axes: What is the latent structure. Is the model explicit or implicit. Is training likelihood-based, adversarial, denoising-based, or transport-based. How is sampling performed.

If these questions can be answered for a new paper, then the student has already understood the architecture of the field at a meaningful level.

For students who are less comfortable with the formalism, it is helpful to keep an intuitive image in mind. A generative model is an engine that starts from simple randomness and shapes it until it looks like structured data. The technical challenge is to decide how this shaping should be parameterized and how the parameters can be learned from examples. Everything in the rest of the course can be read as a refinement of this simple idea. Classical references that support the broad statistical viewpoint include {cite}`bishop2006pattern`, while the modern deep learning context is discussed in {cite}`goodfellow2016deep`.